In [1]:
#testing again


import moviepy
from moviepy import *
from moviepy.video import *
import copy
import accelerate
from pathlib import Path
import torch
import numpy as np
import cv2
import matplotlib
import argparse
import os
import time
import pandas as pd
from PIL import Image
from transformers import (
    AutoProcessor,
    RTDetrForObjectDetection,
    VitPoseForPoseEstimation,
)
print(torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
index = 0
OUT_DIR = '../outputs'

while True:
    save =  'Trial - ' + str(index)
    outputPath = Path(f'{OUT_DIR}/{save}')
    if outputPath.is_dir():
        index += 1
        continue
    else:
        os.makedirs(f'{OUT_DIR}/{save}', exist_ok=False)
        OUT_DIR = OUT_DIR + f'/{save}'
        break


model_name = 'usyd-community/vitpose-plus-huge'



#save_name = 'Ivan'#args.input.split(os.path.sep)[-1].split('.')[0]
# Define codec and create VideoWriter object.

# Load detector.
person_image_processor = AutoProcessor.from_pretrained(
    'PekingU/rtdetr_r50vd_coco_o365'
)
person_model = RTDetrForObjectDetection.from_pretrained(
    'PekingU/rtdetr_r50vd_coco_o365', device_map=device
)
# Load ViTPose.
#print(f"Pose Model: {'../models/VitPose-s_RePoGen.pth'}")
image_processor = AutoProcessor.from_pretrained(model_name)
model = VitPoseForPoseEstimation.from_pretrained(pretrained_model_name_or_path=model_name, device_map=device)


#This is sadly chatted :(
# modelCustom = torch.load('../Models/VitPose-s_RePoGen.pth', device, weights_only=False)
# model.load_state_dict(modelCustom)
# image_processor.load_state_dict(modelCustom)
# image_processor.eval()
# model.eval()

c:\Users\danre\Documents\GitHub\SwimVisionProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
import math


edges = [
    (0, 1), (0, 2), (2, 4), (1, 3), (6, 8), (8, 10),
    (5, 7), (7, 9), (5, 11), (11, 13), (13, 15), (6, 12),
    (12, 14), (14, 16), (5, 6), (11, 12)
]

def lowToHigh(inputList):
    if (inputList[0] < inputList[1]):
        return inputList
    else:
        return (inputList[1], inputList[0])

def detect_objects(image):
    """
    :param image: Image in PIL image format.
    Returns:
        person_boxes: Bboxes of persons in [x, y, w, h] format.
    """
    det_time_start = time.time()
    inputs = person_image_processor(
        images=image, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        outputs = person_model(**inputs)
    target_sizes = torch.tensor([(image.height, image.width)])
    
    results = person_image_processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=0.3
    )
    det_time_end = time.time()
    det_fps = 1 / (det_time_end-det_time_start)
    
    # Extract the first result, as we can pass multiple images at a time.
    result = results[0]
    
    # In COCO dataset, humans labels have index 0.
    person_boxes_xyxy = result['boxes'][result['labels'] == 0]
    person_boxes_xyxy = person_boxes_xyxy.cpu().numpy()
    
    # Convert boxes from (x1, y1, x2, y2) to (x1, y1, w, h) format.
    person_boxes = person_boxes_xyxy.copy()
    person_boxes[:, 2] = person_boxes[:, 2] - person_boxes[:, 0]
    person_boxes[:, 3] = person_boxes[:, 3] - person_boxes[:, 1]
    # print(type(person_boxes))
    # print(person_boxes[0])
    # print(type(person_boxes[0]))
    # print(person_boxes)

    # print("")
    # print(type(person_boxes.item(0)))
    # print(person_boxes.item(0))
    # print("")
    # print(type(np.array([person_boxes[0]])))
    # print(np.array([person_boxes[0]]))

    #return person_boxes, det_fps
    return np.array([person_boxes[0]]), det_fps

def getBB(person_boxes):
    if person_boxes.any():
        box = person_boxes[0].tolist()
        box = [int(x) for x in box]
        return box
    else:
        return []

def drawBB(person_boxes, frame):
    box = getBB(person_boxes)
    if box:
        cv2.rectangle(frame, (box[0], box[1]), (box[0] + box[2], box[1] + box[3]), (0,0,255), 2)
    return frame


def detect_pose(image, person_boxes):
    """
    :param image: Image in PIL image format.
    :param person_bboxes: Batched person boxes in [[x, y, w, h], ...] format.
    """
    pose_time_start = time.time()
    inputs = image_processor(
        image, boxes=[person_boxes], return_tensors='pt'
    ).to(device)
    
    dataset_index = torch.tensor([0], device=device) # must be a tensor of shape (batch_size,)
    if len(person_boxes) != 0:
        if 'plus' in model_name:
            with torch.no_grad():
                outputs = model(**inputs, dataset_index=dataset_index)
        else:
            with torch.no_grad():
                outputs = model(**inputs)
        
        pose_results = image_processor.post_process_pose_estimation(
            outputs, boxes=[person_boxes]
        )
    pose_time_end = time.time()
    pose_fps = 1 / (pose_time_end-pose_time_start)
    if len(person_boxes) == 0:
        return [], pose_fps
    image_pose_result = pose_results[0]
    #THIS IS A SANITY CHECK FOR THE MODEL

    realPoints = getKeypoints(image_pose_result)
    if (abs(slope(realPoints[5], realPoints[11]) > 2)): #More sanity check measures have to be made in the future if I don't get a better model
        image_pose_result = []
    return image_pose_result, pose_fps

body_edges = [(5, 9), (5, 11), (11, 13), (13, 15)] #THESE NEED TO BE LOWER NUMBER FIRST FOR ANGLE OPERATIONS TO WORK

def draw_keypoints_modded(keypoints, frame):
    for p in range(len(keypoints)):
        if p in [5, 9, 11, 13, 15]:
            # draw the keypoints
            cv2.circle(frame, (int(keypoints[p][0]), int(keypoints[p][1])), 
                        3, (0, 0, 255), thickness=-1, lineType=cv2.FILLED)
            # uncomment the following lines if you want to put keypoint number
            cv2.putText(frame, f'{p}', (int(keypoints[p][0]+10), int(keypoints[p][1]-5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
        for ie, e in enumerate(body_edges):
            #if ((set([5, 7, 9, 11, 13, 15]) & set(e)) and not 12 in e and not 6 in e):
                # get different colors for the edges
            rgb = matplotlib.colors.hsv_to_rgb([
                ie/float(len(edges)), 1.0, 1.0
            ])
            rgb = rgb*255
            # join the keypoint pairs to draw the skeletal structure
            cv2.line(frame, (int(keypoints[e[0]][0]), int(keypoints[e[0]][1])),
                    (int(keypoints[e[1]][0]), int(keypoints[e[1]][1])),
                    tuple(rgb), 1, lineType=cv2.LINE_AA)
    return frame




# def drawPersonBoxes(frame, boxes):
#     print(boxes)
#     for box in boxes:
#         cv2.rectangle(frame, (box[0], box[1]), (box[0] + box[2], box[1] + box[3]), (0,0,255), 2)
#     return frame

#Whole skeleton
# body_edges = [(6, 8), (8, 10),
#     (5, 7), (7, 9), (5, 11), (11, 13), (6, 12), (13, 15),
#     (12, 14), (14, 16), (5, 6), (11, 12)]

#Underwaters from left only



#https://stackoverflow.com/questions/14066933/direct-way-of-computing-the-clockwise-angle-between-two-vectors
# [left, vertex, right]
# Format needs to be this way because it calculates clockwise angle, not minimum angle
angles =[[9, 5, 11], [5, 11, 13], [15, 13, 11]] 

def getAngles(points):
    angleList = []
    if points:
        for i in range(len(angles)):
            # print(points[angles[i][0]])
            # print(points[angles[i][1]])
            # print(points[angles[i][2]])

            x1 = points[angles[i][0]][0] - points[angles[i][1]][0]
            y1 = points[angles[i][0]][1] - points[angles[i][1]][1]
            x2 = points[angles[i][2]][0] - points[angles[i][1]][0]
            y2 = points[angles[i][2]][1] - points[angles[i][1]][1]
            # print(f'{x1}, {y1}    {x2}, {y2}')
            pointCombo = [[x1, y1], [x2, y2]]
            # print(pointCombo)
            # print(calcAngle(pointCombo))
            angleList.append(calcAngle(pointCombo))
    return angleList


def calcAngle(pointSet):
    p1, p2 = pointSet
    dot = (p1[0] * p2[0]) + (p1[1] * p2[1])
    det = (p1[0] * p2[1]) - (p1[1] * p2[0]) 
    rawAngle = math.atan2(-det, -dot) + math.pi 
    return round(math.degrees(rawAngle),2)

print(calcAngle([[-172, 29],[163, -34]]))

print(calcAngle([[0.7, -.7], [-0.7, -0.7]]))


samplePoints = [(12, -34), #0
(-8, 41), #1
(25, 25), #2
(-47, -10), #3
(3, 19), #4
(33, -4), #5
(-21, -39), #6
(45, 12), #7
(-15, 28), #8
(0, -48), #9
(37, 37), #10
(39, -55), #11
(18, -22), #12
(-1, -1), #13
(40, -40), #14
(-35, 15), #15
(10, 48)] #16

print(getAngles(samplePoints))




def drawAngles(frame, angleSet, points):
    if points:
        #points = switchOpenCVNormal( image, points)
        # for point in points:
        #     print(point)
        for i in range(len(angleSet)):
            leftIndex = angles[i][2]
            vertexIndex = angles[i][1]
            rightIndex = angles[i][0]

            leftPoint = points[leftIndex]
            vertexPoint = points[vertexIndex]
            rightPoint = points[rightIndex]

            normalizedLeftPoint = [leftPoint[0]- vertexPoint[0], leftPoint[1]-vertexPoint[1]]
            normalizedRightPoint = [rightPoint[0]- vertexPoint[0], rightPoint[1]-vertexPoint[1]]
            extraPointSet = [[1,0], normalizedLeftPoint]
            textAngle = (angleSet[i]/2) + calcAngle(extraPointSet)
            ellipseStartAngle = calcAngle(extraPointSet)
            ellipseEndAngle = calcAngle(extraPointSet) + angleSet[i] 

            cv2.ellipse(frame, (vertexPoint[0], vertexPoint[1]), (30, 30), 0, ellipseStartAngle, ellipseEndAngle, (0,0,255), 2)

            # print(f"Three points: {leftPoint}, {vertexPoint}, {rightPoint}")
            # print(f"Initial Angle: {angleSet[i]}")
            # print(f"Absolute Angle: {absoluteAngle}")

            drawPoint = [vertexPoint[0] + math.cos(math.radians(textAngle))*70 -20, vertexPoint[1] + math.sin(math.radians(textAngle))*70]

            cv2.putText(frame, f'{round(angleSet[i], 2)}', (int(drawPoint[0]), int(drawPoint[1])), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (240, 19, 255), 2) # I converted to left-hand coordinates here
    return frame



def drawAngleDiffs(frame, angleSet, diffAngleSet, points):
    if points and diffAngleSet:
        #points = switchOpenCVNormal( image, points)
        # for point in points:
        #     print(point)

        for i in range(len(angleSet)):
            leftIndex = angles[i][2]
            vertexIndex = angles[i][1]
            rightIndex = angles[i][0]

            leftPoint = points[leftIndex]
            vertexPoint = points[vertexIndex]
            rightPoint = points[rightIndex]

            normalizedLeftPoint = [leftPoint[0]- vertexPoint[0], leftPoint[1]-vertexPoint[1]]
            normalizedRightPoint = [rightPoint[0]- vertexPoint[0], rightPoint[1]-vertexPoint[1]]
            extraPointSet = [[1,0], normalizedLeftPoint]
            textAngle = (angleSet[i]/2) + calcAngle(extraPointSet)
            ellipseStartAngle = calcAngle(extraPointSet)
            ellipseEndAngle = calcAngle(extraPointSet) + angleSet[i] 

            cv2.ellipse(frame, (vertexPoint[0], vertexPoint[1]), (30, 30), 0, ellipseStartAngle, ellipseEndAngle, (0,0,255), 2)

            # print(f"Three points: {leftPoint}, {vertexPoint}, {rightPoint}")
            # print(f"Initial Angle: {angleSet[i]}")
            # print(f"Absolute Angle: {absoluteAngle}")
            yOffset = 0
            
            anglePoint = [vertexPoint[0] + math.cos(math.radians(textAngle))*70 -20, vertexPoint[1] + math.sin(math.radians(textAngle))*70]
            if (math.sin(math.radians(textAngle)) > 0):
                yOffset = 30
            else:
                yOffset = -30
            cv2.putText(frame, f'{round(angleSet[i], 2)}', (int(anglePoint[0]), int(anglePoint[1])), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (240, 19, 255), 2)
            if (diffAngleSet[i] < 0):
                cv2.putText(frame, f'({round(diffAngleSet[i], 2)})', (int(anglePoint[0])+10, int(anglePoint[1]) + yOffset), cv2.FONT_HERSHEY_SIMPLEX, 0.59, (23, 122, 5), 2)
            else:
                cv2.putText(frame, f'({round(diffAngleSet[i], 2)})', (int(anglePoint[0])+10, int(anglePoint[1]) + yOffset), cv2.FONT_HERSHEY_SIMPLEX, 0.59, (0, 0, 255), 2)
    return frame


pointOfFixation = 17

def cacheOffsets(points):
    xDistance = getBodyWidth(points)
    leftDistance = abs(xDistance- abs(points[15][0] - points[pointOfFixation][0]))
    rightDistance = abs(xDistance - abs(points[pointOfFixation][0]-points[9][0]))
    return [[leftDistance, rightDistance], [int(xDistance*.3), int(xDistance*.3)]]

def cacheOffsets(points, bb):

    xDistance = bb[2]
    leftDistance = abs(xDistance- abs(points[15][0] - points[pointOfFixation][0]))
    rightDistance = abs(xDistance - abs(points[pointOfFixation][0]-points[9][0]))
    vertDistance = bb[3]
    return [[leftDistance, rightDistance], [vertDistance]]

def distance(point1, point2):
    return round(math.sqrt((math.pow(point1[0] - point2[0], 2) + math.pow(point1[1]- point2[1], 2))))

print(distance([1, 3], [7, 6]))

def getBodyWidth(points):
    sum = 0
    for pair in body_edges:
        sum += (distance(points[pair[0]], points[pair[1]]))
    return sum

def cropImage(frame, box, points, offsets, frameHeight, frameWidth):
    #buffer = int(offsets[1][0] * 1.2)
    buffer = 50
    heightFactor = box[3] / int(offsets[1][0])
    #heightFactor = box[2] / (offsets[0][0] + offsets[0][1])

    heightPercentile = (points[pointOfFixation][1] - box[1] )/ box[3]
    heightFactorLow = heightFactor * heightPercentile
    heightFactorHigh = heightFactor * (1-heightPercentile)


    lowX = int(points[pointOfFixation][0] - offsets[0][0] * heightFactor - buffer)
    highX = int(points[pointOfFixation][0] + offsets[0][1] * heightFactor + buffer)
    lowY = int(points[pointOfFixation][1] - offsets[1][0] * heightFactorLow - buffer)
    highY = int(points[pointOfFixation][1] + offsets[1][0] * heightFactorHigh + buffer)

    origin = (max(lowX, 0), max(lowY, 0))
    terminal = (min(highX, frameWidth), min(highY, frameHeight))
    print(origin)
    print(terminal)

    points = [origin, terminal]
    croppedImage = frame[origin[1]:terminal[1], origin[0]: terminal[0]]
    croppedImage = cv2.resize(croppedImage, (frameWidth, frameHeight), interpolation=cv2.INTER_CUBIC)
    return croppedImage
        
# def cropImage(frame, box, frameHeight, frameWidth):
#     coords = [box[0], box[1], box[0] + box[2], box[1] + box[3]]
#     buffer = 50
#     lowX = coords[0] - buffer
#     highX = coords[2] + buffer
#     lowY = coords[1] - buffer
#     highY = coords[3] + buffer

#     origin = (max(lowX, 0), max(lowY, 0))
#     terminal = (min(highX, frameWidth), min(highY, frameHeight))

#     points = [origin, terminal]
#     croppedImage = frame[origin[1]:terminal[1], origin[0]: terminal[0]]
#     croppedImage = cv2.resize(croppedImage, (frameWidth, frameHeight), interpolation=cv2.INTER_CUBIC)
#     return croppedImage


def getKeypoints(outputs):
    out = []
    if outputs:
        keypoints = outputs[0]['keypoints'].cpu().detach().numpy()
        keypoints = keypoints[:, :].reshape(-1, 2)
        for p in range(keypoints.shape[0]):
            out.append([int(keypoints[p, 0]), int(keypoints[p, 1])])
        out.append(getAverageCoord(out[5], out[11]))
    return out




def switchOpenCVNormal(image, kps):
    newkps = copy.deepcopy(kps)
    #print(newkps == kps)
    width, height = image.size
    for i in range(len(newkps)):
        newkps[i][1] = height - newkps[i][1]
    return newkps

def getAverageCoord(coord1, coord2):
    return [int((coord1[0] + coord2[0])/2), int((coord1[1] + coord2[1])/2)]

def slope(coord1, coord2):
    if coord1[0] - coord2[0] != 0:
        return ((coord1[1] - coord2[1]) / (coord1[0] - coord2[0])) 
    else:
        return 100 #Arbitrarily high slope


177.79
270.0
[43.58, 29.82, 151.73]
7


In [3]:

targetFPS = 30

def processVideo(videoName, flipNeeded):
    os.makedirs(f'{OUT_DIR}/{videoName}', exist_ok=False)
    inputPath = '../Data/' + videoName + '.mp4'
    cap = cv2.VideoCapture(inputPath)
    frame_width = int(cap.get(3))
    print(frame_width)
    frame_height = int(cap.get(4))
    print(frame_height)
    video_fps = round(cap.get(5)) 
    out = cv2.VideoWriter(
        f'{OUT_DIR}/{videoName}/Processed.mkv', 
        cv2.CAP_FFMPEG,
        cv2.VideoWriter_fourcc(*"FFV1"), 
        targetFPS, 
        (1920, 1080), 
    )
    if not cap.isOpened():
        print("Not opening in processing")
        
    framesNeeded = 0
    while cap.isOpened():
        res, frame = cap.read()
        if res:
            if flipNeeded:
                frame = cv2.flip(frame, 1)
            if frame_width != 1920 or frame_height != 1080:
                frame = cv2.resize(frame, (1920, 1080), interpolation=cv2.INTER_LANCZOS4)
            framesNeeded += (targetFPS/video_fps)
            #print(framesNeeded)
            #cv2.imshow("Processing", frame)
            if (framesNeeded % 1 == 0):
                for j in range(int(framesNeeded)):
                    #print("Writing frame")
                    out.write(frame)
                framesNeeded = 0
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            break
    cap.release()
    cv2.destroyAllWindows()
    out.release()



def flipNeededAndOffsets(videoName):
    inputPath = '../Data/' + videoName + '.mp4'
    cap = cv2.VideoCapture(inputPath)
    vidLength = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    if not cap.isOpened():
        print("Not opening")
    curFrame = int(vidLength/2)
    output = False
    offsets = [[10000, 10000], [10000, 10000]]
    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, curFrame)
        res, frame = cap.read()
        if res:
            #cv2.imshow('Frame', frame)
            if frame_width != 1920 or frame_height != 1080:
                frame = cv2.resize(frame, (1920, 1080), interpolation=cv2.INTER_LANCZOS4)
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)
            bboxes, det_fps = detect_objects(image=image)
            bb = getBB(bboxes)
            image_pose_result, pose_fps = detect_pose(image=image, person_boxes=bboxes)
            kps = getKeypoints(image_pose_result)
            normalkps = switchOpenCVNormal(image, kps)
            if normalkps:
                output = normalkps[5][0] > normalkps[11][0]
                offsets = cacheOffsets(kps, bb)
                break
        curFrame = curFrame+1
        if curFrame == vidLength-1:
            break
    cap.release()
    return [output, offsets]
            


def makeMainVideo(videoName, offsets):
    inputPath = f'{OUT_DIR}/{videoName}/Processed.mkv'
    cap = cv2.VideoCapture(inputPath)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    video_fps = round(cap.get(5)) 

    out = cv2.VideoWriter(
        f'{OUT_DIR}/{videoName}/MainVideo.mp4', 
        #cv2.CAP_FFMPEG,
        cv2.VideoWriter_fourcc(*"avc1"),
        video_fps, 
        (frame_width, frame_height), 
    )

    cachedbb = [0,0] * 4
    cachedkps = [0,0] * 17
    # offsets = [[2000, 2000], [2000, 2000]]
    # cachedOffsets = False
    if not cap.isOpened():
        print("Not opening")
        
    while cap.isOpened():
        ret, frame = cap.read()
        if ret:  
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)
            bboxes, det_fps = detect_objects(image=image)
            bb = getBB(bboxes)
            result = drawBB(bboxes, frame)
            image_pose_result, pose_fps = detect_pose(image=image, person_boxes=bboxes)
            kps = getKeypoints(image_pose_result)
            result = draw_keypoints_modded(kps, result)
            normalkps = switchOpenCVNormal(image, kps)
            angle = getAngles(normalkps)
            result = drawAngles(result, angle, kps)
            print(bboxes)

            if bb:
                cachedbb = bb
            if kps:
                cachedkps = kps
            result = cropImage(result, cachedbb, cachedkps, offsets, frame_height, frame_width)
    
            cv2.imshow("Tested Model", result)
            out.write(result)
            # Press `q` to exit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            break
    cap.release()
    cv2.destroyAllWindows()
    out.release()


learnerVid = "learnerUnderwater"
expertVid = "expertUnderwater"

flipRequired, offsets = flipNeededAndOffsets(expertVid)
processVideo(expertVid, flipRequired)
makeMainVideo(expertVid, offsets)


1920
1080
[[1027.1549   563.97864  641.5117   142.83649]]
(1025, 508)
(1766, 765)
[[1001.7143   349.1279   659.1484   362.48373]]
(998, 445)
(1757, 705)
[[958.03595 564.70135 685.2187  153.77423]]
(963, 509)
(1748, 776)
[[943.7699  561.7671  691.0665  160.13623]]
(954, 508)
(1745, 776)
[[923.3896  570.0042  705.02094 157.2132 ]]
(934, 515)
(1739, 787)
[[888.8314 582.4256 725.068  153.3919]]
(907, 525)
(1732, 801)
[[879.63916 582.23346 732.9249  157.25275]]
(900, 525)
(1732, 804)
[[859.6798  578.7608  751.9867  166.31696]]
(879, 522)
(1730, 805)
[[832.45276 587.87805 776.53516 164.88934]]
(847, 528)
(1723, 818)
[[819.7072  590.4001  790.19257 164.90948]]
(832, 530)
(1722, 823)
[[804.48584 591.6024  798.00977 167.02094]]
(824, 531)
(1722, 825)
[[778.20233 585.8525  834.311   179.02917]]
(795, 525)
(1729, 828)
[[769.08685 591.3694  840.6932  176.73358]]
(788, 530)
(1728, 835)
[[751.81323 593.67346 861.1233  178.5393 ]]
(773, 531)
(1734, 841)
[[727.7029  601.7868  885.0978  176.33923]]
(74